# Tugas Praktik — Structured Streaming with Spark
## Case Study A: CoinGecko → PostgreSQL → Kafka → Spark

**Tujuan:** melengkapi kode Spark Structured Streaming untuk membaca event harga crypto dari Kafka, mem-parsing JSON, membersihkan data, melakukan agregasi berdasarkan window waktu, dan menulis hasil ke console atau Parquet.

### Arsitektur studi kasus
1. Producer mengambil harga CoinGecko dan memasukkannya ke PostgreSQL.
2. Producer kedua membaca record baru dari PostgreSQL dan mengirimkannya ke Kafka topic `crypto_stream`.
3. Spark Structured Streaming membaca topic Kafka dan memproses event.

**Penting:** notebook ini mengasumsikan infrastruktur PostgreSQL, Kafka, dan Spark sudah berjalan sesuai konfigurasi kelas. Notebook ini berfokus pada sisi Spark, bukan membuat seluruh infrastrukturnya.

**Petunjuk:** lengkapi semua bagian `# TODO`. Jalankan cell secara berurutan. Jika nama topic, host, port, atau schema di environment kelas berbeda, ikuti konfigurasi yang diberikan pengajar.

## 1. Import library

In [1]:
# TODO 1: import SparkSession
from pyspark.sql import SparkSession

# TODO 2: import fungsi col, from_json, to_timestamp, window, avg, max
from pyspark.sql.functions import col, from_json, to_timestamp, window, avg, max
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

## 2. Membuat SparkSession

Di environment kelas yang menggunakan Spark standalone, master dapat berupa `spark://spark-master:7077`. Jika menjalankan lokal, ganti menjadi `local[*]`.

In [2]:
# TODO 3: lengkapi pembuatan SparkSession
spark = (
    SparkSession.builder
    .appName("CoinGeckoStructuredStreaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

:: loading settings :: url = jar:file:/Users/fahmialfareza/Code/learning/data-engineering/fullstack-bangalore-advanced-data-engineering/mini-projects/6-sparks/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/fahmialfareza/.ivy2.5.2/cache
The jars for the packages stored in: /Users/fahmialfareza/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b5d865b-6332-457b-90ec-cdb1f58c2024;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in cen

:: resolution report :: resolve 165ms :: artifacts dl 4ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf   

26/11/09 13:33:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.2.0
Master: local[*]


## 3. Mendefinisikan schema payload Kafka

Payload dari producer Case A berbentuk JSON. Untuk latihan ini, kita mengharapkan field `coin`, `price`, dan `time`.

Contoh payload:
`{"coin": "bitcoin", "price": 67250.0, "time": "2026-10-22 10:00:00"}`

Jika producer di kelas hanya mengirim `coin` dan `price`, kolom `time` dapat tidak tersedia. Untuk window aggregation, event time harus tersedia atau ditambahkan pada producer.

In [3]:
# TODO 4: lengkapi schema dengan:
# coin -> StringType
# price -> DoubleType
# time -> StringType
schema = StructType([
    StructField("coin", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("time", StringType(), True)
])

## 4. Membaca data dari Kafka

Spark membaca pesan Kafka sebagai DataFrame dengan kolom metadata dan kolom `value` berbentuk binary. Kita akan mem-parsing `value` sebagai JSON pada langkah berikutnya.

In [4]:
# TODO 5: buat streaming DataFrame dari Kafka
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "crypto_stream")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 5. Parsing JSON dari kolom `value`

Kolom `value` dari Kafka perlu diubah menjadi string lalu diparsing sesuai schema yang sudah dibuat.

In [5]:
# TODO 6: parse value sebagai JSON dan keluarkan field di dalamnya
parsed_df = (
    kafka_df
    .select(
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select("data.*")
)

parsed_df.printSchema()

root
 |-- coin: string (nullable = true)
 |-- price: double (nullable = true)
 |-- time: string (nullable = true)



## 6. Data quality dan event time

Tugas:
- Ubah kolom `time` menjadi timestamp bernama `event_time`.
- Pertahankan hanya harga yang lebih besar dari nol.
- Buang record yang tidak memiliki `event_time`.

In [6]:
# TODO 7: konversi time menjadi event_time
clean_df = (
    parsed_df
    .withColumn("event_time", to_timestamp(col("time")))
    .filter(col("price") > 0)
    .filter(col("event_time").isNotNull())
)

clean_df.printSchema()

root
 |-- coin: string (nullable = true)
 |-- price: double (nullable = true)
 |-- time: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## 7. Window aggregation

Tugas: hitung rata-rata dan harga tertinggi untuk setiap coin pada window 1 menit. Watermark 2 menit digunakan untuk membantu mengelola state dan event yang datang terlambat.

In [7]:
# TODO 8: buat agregasi berdasarkan window dan coin
agg_df = (
    clean_df
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("coin")
    )
    .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price")
    )
)

agg_df.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- coin: string (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- max_price: double (nullable = true)



## 8. Menjalankan query ke console

Gunakan output mode yang sesuai untuk agregasi window. Dalam contoh ini kita menggunakan `append`, sehingga hasil window ditulis ketika dianggap final berdasarkan semantik event time dan watermark.

In [8]:
# TODO 9: lengkapi writeStream
query = (
    agg_df.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("checkpointLocation", "checkpoint/coingecko_console")
    .start()
)

# Ketika ingin menghentikan query, jalankan:
# query.stop()

26/11/09 13:33:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 9. Memantau status query

Jalankan cell ini saat query aktif untuk memeriksa query yang berjalan.

In [9]:
# TODO 10: tampilkan daftar streaming query yang aktif
active_queries = spark.streams.active
print("Active queries:", len(active_queries))
for active_query in active_queries:
    print(active_query.name, active_query.id, active_query.status)

query.processAllAvailable()
query.stop()
print("Console query stopped:", not query.isActive)

Active queries: 1
None 64f72bf0-fe08-43e0-8d4c-ce4ef6df16ac {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/11/09 13:33:03 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+----+---------+---------+
|window|coin|avg_price|max_price|
+------+----+---------+---------+
+------+----+---------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+--------+---------+---------+
|window                                    |coin    |avg_price|max_price|
+------------------------------------------+--------+---------+---------+
|{2026-10-06 20:23:00, 2026-10-06 20:24:00}|bitcoin |86096.0  |86096.0  |
|{2026-10-06 20:18:00, 2026-10-06 20:19:00}|ethereum|2712.17  |2712.17  |
|{2026-10-06 20:19:00, 2026-10-06 20:20:00}|ethereum|2711.23  |2712.17  |
|{2026-10-06 20:32:00, 2026-10-06 20:33:00}|bitcoin |86019.0  |86019.0  |
|{2026-10-06 20:24:00, 2026-10-06 20:25:00}|ethereum|2712.42  |2712.42  |
|{2026-10-06 20:28:00, 2026-10-06 20:29:00}|bitcoin |86063.0  |86063.0  |
|{2026-10-06 20:25:00, 2026-10-06 20:26:00}|bitcoin |86076.0  |86076.0  |
|{2026-10-06 20:31:00, 2026-10-06 20:32:00}|bitcoin |86013.0  |86013.0  |
|{2026-10-06 20:31:00, 2026-10-06 20:32:00}|ethereum|2710.18  |2710.18  |
|{2026-10-06 20

26/11/09 13:33:06 WARN DAGScheduler: Failed to cancel job group f7d47fe1-b10e-4377-ac18-8c09e91ab836. Cannot find active jobs for it.
26/11/09 13:33:06 WARN DAGScheduler: Failed to cancel job group f7d47fe1-b10e-4377-ac18-8c09e91ab836. Cannot find active jobs for it.


## 10. Tantangan lanjutan — simpan hasil ke Parquet

Setelah query console dihentikan, buat query baru yang menulis agregasi ke Parquet. Gunakan checkpoint berbeda dari query console.

**Catatan:** lokasi `s3a://...` hanya dapat digunakan jika konfigurasi Hadoop S3A dan MinIO tersedia di cluster. Untuk latihan lokal, gunakan folder lokal seperti `output/coingecko_agg`.

In [10]:
# TODO 11: tulis agg_df ke Parquet dengan checkpoint terpisah
parquet_query = (
    agg_df.writeStream
    .format("parquet")
    .option("path", "output/coingecko_agg")
    .option("checkpointLocation", "checkpoint/coingecko_parquet")
    .outputMode("append")
    .start()
)

parquet_query.processAllAvailable()
parquet_query.stop()
print("Parquet query stopped:", not parquet_query.isActive)

26/11/09 13:33:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/09 13:33:06 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


Parquet query stopped: True


26/11/09 13:33:08 WARN DAGScheduler: Failed to cancel job group 5f09f276-a144-46b4-a331-212cc0e06125. Cannot find active jobs for it.
26/11/09 13:33:08 WARN DAGScheduler: Failed to cancel job group 5f09f276-a144-46b4-a331-212cc0e06125. Cannot find active jobs for it.


## 11. Troubleshooting

Periksa hal berikut jika hasil tidak muncul:

- Spark dan Kafka dapat diakses dari environment notebook.
- Topic `crypto_stream` sudah dibuat dan producer mengirim pesan.
- `startingOffsets="latest"` hanya membaca pesan baru setelah query mulai berjalan; untuk data lama, pertimbangkan `earliest` saat memulai query baru dengan checkpoint yang sesuai.
- JSON payload sesuai schema.
- Untuk window aggregation, field waktu tersedia dan berhasil dikonversi menjadi timestamp.
- Checkpoint untuk query berbeda tidak menggunakan folder yang sama.
- Query console sebelumnya sudah dihentikan sebelum menjalankan query baru pada sumber dan checkpoint yang sama.
- Untuk output Parquet, pastikan path dapat ditulis dan mode output didukung oleh query.

**Catatan penting:** API CoinGecko bukan sumber push streaming secara langsung. Pada Case A, data diambil secara berkala oleh producer, lalu dialirkan melalui PostgreSQL dan Kafka sebelum diproses Spark.

## 12. Pertanyaan refleksi

1. Apa fungsi `from_json()` pada pipeline ini?
2. Mengapa schema Kafka perlu didefinisikan?
3. Apa perbedaan event time dan waktu pemrosesan?
4. Mengapa `checkpointLocation` untuk console dan Parquet dipisahkan?
5. Apa yang perlu diubah jika producer tidak mengirim field `time`?
6. Jelaskan alur data dari CoinGecko sampai ke hasil agregasi Spark.

Tuliskan jawaban pada cell berikut.

In [11]:
# Jawaban 1: from_json() mengubah string JSON pada value Kafka menjadi struct bertipe sesuai schema agar setiap field dapat diproses sebagai kolom.
# Jawaban 2: Schema memberi tipe data eksplisit, membuat parsing konsisten, memungkinkan validasi, dan membantu Spark mengoptimalkan eksekusi.
# Jawaban 3: Event time adalah waktu kejadian pada sumber, sedangkan processing time adalah waktu event diproses oleh Spark.
# Jawaban 4: Setiap query memiliki progress dan state sendiri; checkpoint terpisah mencegah konflik offset serta state antara sink console dan Parquet.
# Jawaban 5: Producer sebaiknya menambahkan timestamp kejadian. Alternatifnya Spark dapat memakai current_timestamp(), tetapi itu mengubah semantik menjadi processing time.
# Jawaban 6: Producer mem-poll CoinGecko dan menyimpan harga ke PostgreSQL; publisher membaca baris baru ke Kafka; Spark membaca crypto_stream, parsing dan membersihkan event, lalu membuat agregasi window untuk sink console atau Parquet.

## Checklist sebelum dikumpulkan

- [x] Semua bagian TODO sudah dilengkapi.
- [x] Schema sesuai dengan payload Kafka yang benar-benar dikirim.
- [x] Parsing JSON dan filter data berhasil.
- [x] Window aggregation berhasil didefinisikan.
- [x] Query console berhasil dijalankan dan dihentikan.
- [x] Checkpoint query ditentukan dengan benar.
- [x] Jawaban refleksi ditulis.

**Pengumpulan:** kumpulkan file `.ipynb` yang sudah dikerjakan beserta screenshot/output penting bila diminta pengajar.